# Limpeza de `pesquisa_saida.csv`

**Objetivo:** gerar a pesquisa de saída pronta para análise em `dados/processados/pesquisa_saida_limpa.parquet`.

**Regras aplicadas** (decisões de 07/11/2026; o diagnóstico está no notebook 18):

| Passo | Regra |
|---|---|
| 1 | `respondido_em` como data (`data_resposta`); comentário vazio vira ausente; `comentario_norm` em minúsculas |
| 2 | `tema_comentario` por palavras-chave (provisório, até a extração com IA) e `motivo_ajustado`: `objetivo_cumprido` quando o comentário fala em prova ou objetivo cumprido, senão o `motivo_principal` |
| 3 | `resposta_mais_recente`: a resposta que vale por conta (2 contas responderam duas vezes) |

**Atenção ao usar:** o motivo declarado cobre só quem cancelou pelo site e quis responder (10% dos fins de assinatura de
junho a setembro): não generalize para todos os que saem. O `motivo_principal` contradiz o comentário em 54% dos comentários
com tema comparável: use o `motivo_ajustado` e o comentário juntos.

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.

In [1]:
ARQUIVO = "pesquisa_saida.csv"
NOME_SAIDA = "pesquisa_saida_limpa.parquet"

import hashlib
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = [ARQUIVO]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df, alteradas=0):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df),
                     "alteradas": int(alteradas)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas, {int(alteradas):,} alteradas)")
    return df

SEM_CONTEUDO = {"nada", "ok", "x", ".", "-", "asdf", "kkkk", "sla", "nao sei", "não sei", "nan", "null", "none", "n/a"}

def sem_conteudo(s):
    t = s.str.strip().str.lower()
    so_simbolo = t.ne("") & ~t.str.contains(r"[a-z0-9à-ú]", regex=True)
    return t.isin(SEM_CONTEUDO) | so_simbolo

usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro", "canal_aquisicao"]).set_index("user_id")

## 0. Leitura

In [2]:
pes = pd.read_csv(BRUTOS / ARQUIVO, dtype=str, keep_default_na=False)
assert len(pes) == 643 and pes["resposta_id"].is_unique
base = registrar("0. Arquivo bruto", "pesquisa_saida.csv sem filtro", pes.copy())

0. Arquivo bruto: 643 linhas (0 removidas, 0 alteradas)


## 1. Data e comentário

In [3]:
base["data_resposta"] = pd.to_datetime(base["respondido_em"], format="%Y-%m-%d")
assert base["data_resposta"].between("2026-06-01", CORTE).all()
vazio = base["comentario"].str.strip() == ""
base["comentario"] = base["comentario"].str.strip().where(~vazio, None)
base["comentario_norm"] = base["comentario"].str.lower()
print(f"comentários vazios que viram ausente: {vazio.sum()}")
base = registrar("1. Data e comentário", "data_resposta; comentário vazio vira ausente; comentario_norm em minúsculas",
                 base, vazio.sum())

comentários vazios que viram ausente: 183
1. Data e comentário: 643 linhas (0 removidas, 183 alteradas)


## 2. Tema do comentário e motivo ajustado

Mesmas palavras-chave do diagnóstico (vale o primeiro tema que casar). O tema é provisório: serve para conferir e
para o `motivo_ajustado`, até a extração com IA generativa.

In [4]:
TEMAS = {
    "objetivo cumprido (prova)": r"prova|objetivo cumprido",
    "outro app": r"strava|outro app|nike|garmin connect|outra plataforma",
    "saúde": r"lesão|lesao|médico|medico|cirurgia|joelho|saúde|saude|grávida|gravida",
    "vida pessoal": r"trabalho|filho|mudei|mudança|viagem|faculdade|casamento",
    "problema técnico": r"bug|gps|trav|erro|atualiza|não funciona|nao funciona",
    "preço": r"caro|preço|preco|dinheiro|grana|desconto|pagar",
    "não uso / perdi a vontade": r"usando|vontade|chata|tempo|mesma coisa|mesmo treino|enjoei|cansei",
}
coment = base["comentario_norm"].fillna("")
tema = pd.Series(None, index=base.index, dtype=object)
for nome, rx in TEMAS.items():
    tema = tema.where(tema.notna() | ~coment.str.contains(rx, regex=True), nome)
base["tema_comentario"] = tema
prova = base["tema_comentario"] == "objetivo cumprido (prova)"
base["motivo_ajustado"] = base["motivo_principal"].where(~prova, "objetivo_cumprido")
display(pd.crosstab(base["motivo_principal"], base["motivo_ajustado"]))
assert prova.sum() == 158
base = registrar("2. Tema e motivo ajustado", "objetivo_cumprido quando o comentário fala em prova ou objetivo cumprido",
                 base, prova.sum())

motivo_ajustado,nao_uso,objetivo_cumprido,outro,outro_app,preco,problema_tecnico,saude
motivo_principal,,,,,,,
nao_uso,76,5,0,0,0,0,0
outro,0,85,138,0,0,0,0
outro_app,0,3,0,31,0,0,0
preco,0,65,0,0,199,0,0
problema_tecnico,0,0,0,0,0,27,0
saude,0,0,0,0,0,0,14


2. Tema e motivo ajustado: 643 linhas (0 removidas, 158 alteradas)


## 3. Resposta mais recente por conta

In [5]:
ordem = base.sort_values(["user_id", "data_resposta", "resposta_id"])
base["resposta_mais_recente"] = ~ordem.duplicated("user_id", keep="last").reindex(base.index)
assert (~base["resposta_mais_recente"]).sum() == 2 and base.loc[base["resposta_mais_recente"], "user_id"].is_unique
base = registrar("3. Resposta mais recente", "marca a última resposta de cada conta", base)

3. Resposta mais recente: 643 linhas (0 removidas, 0 alteradas)


## 4. Conferências

In [6]:
conferencias = {
    "linhas": (len(base), 643),
    "contas": (int(base["user_id"].nunique()), 641),
    "contas fora de usuarios_limpa": (int((~base["user_id"].isin(usuarios.index)).sum()), 0),
    "motivo fora da lista (com objetivo_cumprido)": (
        int((~base["motivo_ajustado"].isin({"preco", "nao_uso", "problema_tecnico", "outro_app", "saude", "outro", "objetivo_cumprido"})).sum()), 0),
    "respostas que valem por conta": (int(base["resposta_mais_recente"].sum()), 641),
}

In [7]:
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()

,obtido,esperado,ok
linhas,643,643,True
contas,641,641,True
contas fora de usuarios_limpa,0,0,True
motivo fora da lista (com objetivo_cumprido),0,0,True
respostas que valem por conta,641,641,True


## 5. Saídas e integridade dos brutos

In [8]:
COLUNAS = ["resposta_id", "user_id", "data_resposta", "motivo_principal", "motivo_ajustado", "comentario", "comentario_norm",
           "tema_comentario", "resposta_mais_recente"]
limpa = base[COLUNAS].reset_index(drop=True)

for col in limpa.columns[limpa.dtypes == object]:
    limpa[col] = limpa[col].astype(object).where(limpa[col].notna(), None)   # ausente sempre como None

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / NOME_SAIDA
limpa.to_parquet(destino, index=False)
relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)
pd.DataFrame(registro).to_csv(SAIDAS / f"limpeza_{Path(ARQUIVO).stem}_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\pesquisa_saida_limpa.parquet: 643 linhas x 9 colunas


,tipo,ausentes
resposta_id,object,0
user_id,object,0
data_resposta,datetime64[ns],0
motivo_principal,object,0
motivo_ajustado,object,0
comentario,object,183
comentario_norm,object,183
tema_comentario,object,264
resposta_mais_recente,bool,0


,passo,regra,linhas,removidas,alteradas
0,0. Arquivo bruto,pesquisa_saida.csv sem filtro,643,0,0
1,1. Data e comentário,data_resposta; comentário vazio vira ausente; comentario_norm em m...,643,0,183
2,2. Tema e motivo ajustado,objetivo_cumprido quando o comentário fala em prova ou objetivo cu...,643,0,158
3,3. Resposta mais recente,marca a última resposta de cada conta,643,0,0


dados/brutos/ inalterado: True
